# Assistant biomédical à verdict calibré — Jour 2
### Fine-tuning LoRA de PubMedBERT sur la classification yes/no/maybe

Ce notebook suppose que le **Jour 1** a été exécuté (données chargées, nettoyées,
échantillonnées et sauvegardées dans `/content/drive/MyDrive/assistant_biomedical/data/`).

⚠️ **Point important sur les données** : `pqa_artificial` ne contient quasiment que
des labels *yes/no* — le label *maybe* n'existe réellement que dans les 1 000 exemples
experts (`pqa_labeled`). Pour que le modèle apprenne les trois classes, on réinjecte
une partie de `pqa_labeled` dans le train/val, et on réserve le reste comme **test
final held-out**, jamais vu pendant l'entraînement.


## 1. Setup, GPU et dépendances

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
!pip install -q transformers peft accelerate evaluate scikit-learn datasets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import numpy as np
import torch

PROJECT_DIR = "/content/drive/MyDrive/assistant_biomedical"
DATA_DIR = f"{PROJECT_DIR}/data"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
RESULTS_DIR = f"{PROJECT_DIR}/results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)


## 2. Chargement des splits du Jour 1

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_artificial = load_json(f"{DATA_DIR}/train.json")
val_artificial = load_json(f"{DATA_DIR}/val.json")
test_expert_full = load_json(f"{DATA_DIR}/test_expert.json")

print("Train artificial :", len(train_artificial))
print("Val artificial   :", len(val_artificial))
print("Test expert (1000, PubMedQA labeled) :", len(test_expert_full))


In [ ]:
from collections import Counter

print("Distribution labels — train artificial :", Counter([x['label'] for x in train_artificial]))
print("Distribution labels — test expert complet :", Counter([x['label'] for x in test_expert_full]))


## 3. Correction du déséquilibre "maybe"

On découpe `test_expert_full` (1 000 exemples) en 3 parts stratifiées :
- **expert_train_exposure** (~60%) → ajouté au train
- **expert_val** (~20%) → ajouté à la validation
- **expert_final_test** (~20%) → réservé, jamais utilisé avant l'évaluation finale (Jour 3-4)


In [ ]:
def stratified_split(data, ratios=(0.6, 0.2, 0.2), seed=SEED):
    random.seed(seed)
    by_label = {"yes": [], "no": [], "maybe": []}
    for ex in data:
        by_label[ex["label"]].append(ex)

    part1, part2, part3 = [], [], []
    for label, items in by_label.items():
        random.shuffle(items)
        n = len(items)
        n1 = round(n * ratios[0])
        n2 = round(n * ratios[1])
        part1.extend(items[:n1])
        part2.extend(items[n1:n1+n2])
        part3.extend(items[n1+n2:])

    random.shuffle(part1)
    random.shuffle(part2)
    random.shuffle(part3)
    return part1, part2, part3

expert_train_exposure, expert_val, expert_final_test = stratified_split(test_expert_full)

print("Expert -> train exposure :", len(expert_train_exposure), Counter([x['label'] for x in expert_train_exposure]))
print("Expert -> val            :", len(expert_val), Counter([x['label'] for x in expert_val]))
print("Expert -> final test     :", len(expert_final_test), Counter([x['label'] for x in expert_final_test]))


In [ ]:
# Fusion : artificiel (yes/no majoritairement) + exposition experte (yes/no/maybe)
final_train = train_artificial + expert_train_exposure
final_val = val_artificial + expert_val

random.shuffle(final_train)
random.shuffle(final_val)

print("Train final :", len(final_train), Counter([x['label'] for x in final_train]))
print("Val final   :", len(final_val), Counter([x['label'] for x in final_val]))
print("Test final held-out (jamais entraîné dessus) :", len(expert_final_test))

# Sauvegarde pour traçabilité et pour les jours suivants
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

save_json(final_train, f"{DATA_DIR}/final_train.json")
save_json(final_val, f"{DATA_DIR}/final_val.json")
save_json(expert_final_test, f"{DATA_DIR}/final_test_expert_holdout.json")


## 4. Tokenisation et préparation Hugging Face `datasets`

On utilise `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`, encodeur biomédical
pré-entraîné sur PubMed, bien adapté à un fine-tuning léger sur T4.

In [ ]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LABEL2ID = {"yes": 0, "no": 1, "maybe": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MAX_LENGTH = 384

def to_hf_format(data):
    return {
        "text_pair": [(ex["question"], ex["context"]) for ex in data],
        "label": [LABEL2ID[ex["label"]] for ex in data],
    }

from datasets import Dataset

def build_dataset(data):
    questions = [ex["question"] for ex in data]
    contexts = [ex["context"] for ex in data]
    labels = [LABEL2ID[ex["label"]] for ex in data]
    ds = Dataset.from_dict({"question": questions, "context": contexts, "label": labels})
    return ds

train_ds = build_dataset(final_train)
val_ds = build_dataset(final_val)
test_ds = build_dataset(expert_final_test)

def tokenize_fn(batch):
    return tokenizer(
        batch["question"],
        batch["context"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

columns = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=columns)
val_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)

print(train_ds)


## 5. Poids de classe (déséquilibre résiduel)

Même après réinjection des exemples experts, "maybe" reste minoritaire. On calcule
des poids de classe pour la fonction de perte afin de ne pas laisser le modèle
ignorer cette classe.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_labels_np = np.array([LABEL2ID[ex["label"]] for ex in final_train])
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels_np,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Poids de classe (yes, no, maybe) :", class_weights)


## 6. Modèle + LoRA

Configuration LoRA : rang=8, alpha=16 (comme spécifié au cahier des charges),
appliquée sur les projections `query` et `value` de l'attention BERT.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.to(device)


## 7. Trainer avec perte pondérée par classe

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels") if "labels" in inputs else inputs.pop("label")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
# Le Trainer attend une colonne 'labels' (et non 'label')
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir=f"{CKPT_DIR}/pubmedbert_lora",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-4,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


## 8. Entraînement

⚠️ Sur T4 gratuit, compter environ 30-45 minutes pour 3 epochs sur ~20 600 exemples.
Le meilleur checkpoint (macro-F1 sur validation) est automatiquement restauré à la fin.

In [ ]:
train_result = trainer.train()
print(train_result)


## 9. Sauvegarde de l'adaptateur LoRA sur Drive

In [ ]:
ADAPTER_DIR = f"{CKPT_DIR}/pubmedbert_lora_adapter_final"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adaptateur LoRA sauvegardé dans :", ADAPTER_DIR)


## 10. Évaluation sur le test expert held-out (jamais vu)

C'est la vraie mesure de performance : ce sous-ensemble n'a été utilisé ni pour le
train ni pour la validation.

In [ ]:
eval_results = trainer.evaluate(eval_dataset=test_ds)
print("Résultats sur le test expert held-out :", eval_results)


In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

y_pred_labels = [ID2LABEL[i] for i in y_pred]
y_true_labels = [ID2LABEL[i] for i in y_true]

print(classification_report(y_true_labels, y_pred_labels, digits=3))


## 11. Comparaison avec la baseline zero-shot (Jour 1)

In [ ]:
baseline_predictions = load_json(f"{RESULTS_DIR}/baseline_zero_shot_predictions.json")

baseline_true = [p["true_label"] for p in baseline_predictions]
baseline_pred = [p["predicted_label"] for p in baseline_predictions]

baseline_acc = accuracy_score(baseline_true, baseline_pred)
baseline_f1 = f1_score(baseline_true, baseline_pred, average="macro")

finetuned_acc = eval_results["eval_accuracy"]
finetuned_f1 = eval_results["eval_macro_f1"]

print(f"{'Modèle':30s} {'Accuracy':>10s} {'Macro-F1':>10s}")
print(f"{'Baseline zero-shot (Qwen2.5)':30s} {baseline_acc:>10.4f} {baseline_f1:>10.4f}")
print(f"{'PubMedBERT + LoRA (fine-tuné)':30s} {finetuned_acc:>10.4f} {finetuned_f1:>10.4f}")


In [ ]:
comparison = {
    "baseline_zero_shot": {"accuracy": baseline_acc, "macro_f1": baseline_f1},
    "pubmedbert_lora_finetuned": {"accuracy": finetuned_acc, "macro_f1": finetuned_f1},
}
save_json(comparison, f"{RESULTS_DIR}/day2_comparison_baseline_vs_finetuned.json")
print("Comparaison sauvegardée.")


## 12. Bilan du Jour 2

- ✅ Correction du déséquilibre "maybe" (réinjection stratifiée des exemples experts)
- ✅ Fine-tuning LoRA (r=8, alpha=16) de PubMedBERT avec perte pondérée par classe
- ✅ Adaptateur LoRA sauvegardé sur Drive
- ✅ Évaluation sur test expert held-out + comparaison chiffrée avec la baseline zero-shot

**Prochaine étape (Jour 3)** : calibration (température scaling, ECE), seuil
d'abstention, détection hors périmètre, et construction de l'index FAISS pour
la citation des passages sources.